In [0]:
# Databricks notebook source
# ETL - Squad 3 - ecommerce_rastreamento_entregas
# Camada Silver
# Fluxo: Bronze Delta -> Silver Delta

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# 4. Define nomes, paths, modo de escrita e chave de deduplicação.

BRONZE_TABLE = "ecommerce_rastreamento_entregas"
SILVER_TABLE = "ecommerce_rastreamento_entregas"

BRONZE_PATH = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"
SILVER_PATH = f"{SILVER_BASE_PATH}{SILVER_TABLE}"

SILVER_WRITE_MODE = "overwrite"

KEY_COLUMNS = ["id_rastreamento"]

print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("KEY_COLUMNS:", KEY_COLUMNS)

In [0]:
# 5. Recupera opções de conexão com o ADLS.

adls_options = get_adls_options()

print("Opções ADLS configuradas.")

In [0]:
# 6. Lê a Bronze em Delta.

df_bronze = (
    spark
    .read
    .format("delta")
    .options(**adls_options)
    .load(BRONZE_PATH)
)

df_bronze.printSchema()
display(df_bronze.limit(10))

In [0]:
# 7. Conta registros de entrada.

total_bronze = df_bronze.count()

print(f"Total de registros lidos da Bronze: {total_bronze}")

In [0]:
# 8. Valida casts principais: IDs para inteiro e `dt_evento` para timestamp.

from pyspark.sql.functions import col, to_timestamp, count, when

df_validation = (
    df_bronze
    .withColumn("id_rastreamento_int", col("id_rastreamento").cast("int"))
    .withColumn("id_pedido_ecommerce_int", col("id_pedido_ecommerce").cast("int"))
    .withColumn("id_transportadora_int", col("id_transportadora").cast("int"))
    .withColumn("dt_evento_timestamp", to_timestamp(col("dt_evento")))
)

df_validacao_conversoes = df_validation.select(
    count("*").alias("total_linhas"),

    count(
        when(
            col("id_rastreamento").isNotNull() &
            col("id_rastreamento_int").isNull(),
            True
        )
    ).alias("falhas_id_rastreamento"),

    count(
        when(
            col("id_pedido_ecommerce").isNotNull() &
            col("id_pedido_ecommerce_int").isNull(),
            True
        )
    ).alias("falhas_id_pedido_ecommerce"),

    count(
        when(
            col("id_transportadora").isNotNull() &
            col("id_transportadora_int").isNull(),
            True
        )
    ).alias("falhas_id_transportadora"),

    count(
        when(
            col("dt_evento").isNotNull() &
            col("dt_evento_timestamp").isNull(),
            True
        )
    ).alias("falhas_dt_evento")
)

display(df_validacao_conversoes)

validacao_conversoes = df_validacao_conversoes.collect()[0]

if validacao_conversoes["falhas_id_rastreamento"] > 0:
    raise Exception("Existem valores de id_rastreamento que não podem ser convertidos para integer.")

if validacao_conversoes["falhas_id_pedido_ecommerce"] > 0:
    raise Exception("Existem valores de id_pedido_ecommerce que não podem ser convertidos para integer.")

if validacao_conversoes["falhas_id_transportadora"] > 0:
    raise Exception("Existem valores de id_transportadora que não podem ser convertidos para integer.")

if validacao_conversoes["falhas_dt_evento"] > 0:
    raise Exception("Existem valores de dt_evento que não podem ser convertidos para timestamp.")

print("Validação OK: conversões principais podem ser feitas.")

In [0]:
# 9. Inspeciona os status antes da padronização.

from pyspark.sql.functions import lower, trim

df_status_entrega = (
    df_bronze
    .withColumn("status_entrega_normalizado", lower(trim(col("status_entrega"))))
    .groupBy("status_entrega_normalizado")
    .count()
    .orderBy("status_entrega_normalizado")
)

display(df_status_entrega)

In [0]:
# 10. Aplica regra de padronização de `status_entrega` e identifica status válidos.

from pyspark.sql.functions import lower, trim, regexp_replace, translate, col, count, when

STATUS_VALIDOS = [
    "coletado",
    "em separacao",
    "em transito",
    "saiu para entrega",
    "entregue"
]

df_bronze_status = (
    df_bronze
    .withColumn(
        "status_entrega_normalizado",
        regexp_replace(
            translate(
                lower(trim(col("status_entrega"))),
                "áàâãéêíóôõúç",
                "aaaaeeiooouc"
            ),
            "\\s+",
            " "
        )
    )
    .withColumn(
        "status_valido",
        col("status_entrega_normalizado").isin(STATUS_VALIDOS)
    )
)

df_validacao_status = (
    df_bronze_status
    .groupBy("status_entrega_normalizado", "status_valido")
    .count()
    .orderBy("status_valido", "status_entrega_normalizado")
)

display(df_validacao_status)

total_status_invalidos = (
    df_bronze_status
    .filter(~col("status_valido"))
    .count()
)

print(f"Total de registros com status fora do fluxo: {total_status_invalidos}")
print("Registros com status fora do fluxo ficarão apenas na Bronze.")

In [0]:
# 11. Cria a Silver aplicando casts, filtro de status, deduplicação e `dias_em_transito`.

from pyspark.sql.functions import (
    col,
    trim,
    to_timestamp,
    current_timestamp,
    row_number,
    min as spark_min,
    max as spark_max,
    datediff,
    to_date
)

from pyspark.sql.window import Window

df_silver_base = (
    df_bronze_status
    .withColumn("id_rastreamento_int", col("id_rastreamento").cast("int"))
    .withColumn("id_pedido_ecommerce_int", col("id_pedido_ecommerce").cast("int"))
    .withColumn("id_transportadora_int", col("id_transportadora").cast("int"))
    .withColumn("dt_evento_ts", to_timestamp(col("dt_evento")))
    .filter(col("status_valido"))
    .filter(col("id_rastreamento_int").isNotNull())
)

window_deduplicacao = (
    Window
    .partitionBy("id_rastreamento_int")
    .orderBy(
        col("dt_evento_ts").desc_nulls_last(),
        col("bronze_ingested_at").desc_nulls_last()
    )
)

df_silver_deduplicada = (
    df_silver_base
    .withColumn("rn", row_number().over(window_deduplicacao))
    .filter(col("rn") == 1)
)

# calcular dias_em_transito pelo primeiro e último evento do pedido

window_pedido = Window.partitionBy("id_pedido_ecommerce_int")

df_silver_com_datas = (
    df_silver_deduplicada
    .withColumn(
        "dt_primeiro_evento_pedido",
        spark_min(col("dt_evento_ts")).over(window_pedido)
    )
    .withColumn(
        "dt_ultimo_evento_pedido",
        spark_max(col("dt_evento_ts")).over(window_pedido)
    )
    .withColumn(
        "dias_em_transito",
        datediff(
            to_date(col("dt_ultimo_evento_pedido")),
            to_date(col("dt_primeiro_evento_pedido"))
        )
    )
)

df_silver = (
    df_silver_com_datas
    .select(
        col("id_rastreamento_int").alias("id_rastreamento"),
        col("id_pedido_ecommerce_int").alias("id_pedido_ecommerce"),
        trim(col("codigo_rastreio")).alias("codigo_rastreio"),
        col("id_transportadora_int").alias("id_transportadora"),
        col("status_entrega_normalizado").alias("status_entrega"),
        col("dt_evento_ts").alias("dt_evento"),
        trim(col("observacao")).alias("observacao"),

        col("dias_em_transito"),

        col("bronze_source_file"),
        col("bronze_ingested_at"),

        current_timestamp().alias("silver_processed_at"),

        col("ano").cast("int").alias("ano"),
        col("mes").cast("int").alias("mes")
    )
)

In [0]:
# 12. Valida a Silver em memória antes da escrita.

from pyspark.sql.functions import countDistinct

total_silver = df_silver.count()

total_ids_distintos_validos = (
    df_bronze_status
    .filter(col("status_valido"))
    .filter(col("id_rastreamento").isNotNull())
    .select(col("id_rastreamento").cast("int").alias("id_rastreamento"))
    .distinct()
    .count()
)

duplicados_silver = (
    df_silver
    .groupBy("id_rastreamento")
    .count()
    .filter(col("count") > 1)
    .count()
)

status_invalidos_silver = (
    df_silver
    .filter(~col("status_entrega").isin(STATUS_VALIDOS))
    .count()
)

df_validacao_silver = df_silver.select(
    count("*").alias("total_linhas"),
    count(when(col("id_rastreamento").isNull(), True)).alias("id_rastreamento_nulo"),
    count(when(col("id_pedido_ecommerce").isNull(), True)).alias("id_pedido_ecommerce_nulo"),
    count(when(col("id_transportadora").isNull(), True)).alias("id_transportadora_nulo"),
    count(when(col("status_entrega").isNull(), True)).alias("status_entrega_nulo"),
    count(when(col("dt_evento").isNull(), True)).alias("dt_evento_nulo"),
    count(when(col("ano").isNull(), True)).alias("ano_nulo"),
    count(when(col("mes").isNull(), True)).alias("mes_nulo"),
    count(when(col("silver_processed_at").isNull(), True)).alias("silver_processed_at_nulo"),
    count(when(col("dias_em_transito").isNull(), True)).alias("dias_em_transito_nulo")
)

display(df_validacao_silver)

print(f"Total Bronze: {total_bronze}")
print(f"Total Silver após filtro e deduplicação: {total_silver}")
print(f"IDs distintos válidos na Bronze: {total_ids_distintos_validos}")
print(f"Duplicados na Silver: {duplicados_silver}")
print(f"Status inválidos na Silver: {status_invalidos_silver}")

if total_silver != total_ids_distintos_validos:
    raise Exception("Erro: quantidade da Silver diferente dos IDs válidos distintos da Bronze.")

if duplicados_silver > 0:
    raise Exception("Erro: ainda existem id_rastreamento duplicados na Silver.")

if status_invalidos_silver > 0:
    raise Exception("Erro: existem status fora do fluxo na Silver.")

validacao_silver = df_validacao_silver.collect()[0]

if validacao_silver["id_rastreamento_nulo"] > 0:
    raise Exception("Erro: existem registros com id_rastreamento nulo na Silver.")

if validacao_silver["dt_evento_nulo"] > 0:
    raise Exception("Erro: existem registros com dt_evento nulo na Silver.")

if validacao_silver["ano_nulo"] > 0:
    raise Exception("Erro: existem registros com ano nulo na Silver.")

if validacao_silver["mes_nulo"] > 0:
    raise Exception("Erro: existem registros com mes nulo na Silver.")

print("Validação OK: Silver em memória aprovada.")

In [0]:
# 13. Grava a Silver particionada por `ano` e `mes`.

(
    df_silver
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(SILVER_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(SILVER_PATH)
)

print(f"Dados gravados com sucesso na Silver: {SILVER_PATH}")

In [0]:
# 14. Lê a Silver gravada.

df_silver_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PATH)
)

df_silver_saved.printSchema()

total_silver_saved = df_silver_saved.count()

print(f"Total de registros na Silver gravada: {total_silver_saved}")

display(df_silver_saved.limit(10))

In [0]:
# 15. Valida quantidade, deduplicação e status da Silver persistida.

duplicados_silver_saved = (
    df_silver_saved
    .groupBy("id_rastreamento")
    .count()
    .filter(col("count") > 1)
    .count()
)

status_invalidos_silver_saved = (
    df_silver_saved
    .filter(~col("status_entrega").isin(STATUS_VALIDOS))
    .count()
)

print(f"Total Silver em memória: {total_silver}")
print(f"Total Silver gravada: {total_silver_saved}")
print(f"IDs distintos válidos na Bronze: {total_ids_distintos_validos}")
print(f"Duplicados na Silver gravada: {duplicados_silver_saved}")
print(f"Status inválidos na Silver gravada: {status_invalidos_silver_saved}")

if total_silver_saved != total_silver:
    raise Exception("Erro: quantidade da Silver gravada diferente da Silver em memória.")

if total_silver_saved != total_ids_distintos_validos:
    raise Exception("Erro: quantidade da Silver gravada diferente dos IDs válidos distintos da Bronze.")

if duplicados_silver_saved > 0:
    raise Exception("Erro: existem id_rastreamento duplicados na Silver gravada.")

if status_invalidos_silver_saved > 0:
    raise Exception("Erro: existem status fora do fluxo na Silver gravada.")

print("Validação OK: quantidade, deduplicação e status da Silver gravada conferem.")

In [0]:
# 16. Valida schema final e campos críticos.

from pyspark.sql.functions import count, when, col

colunas_silver = df_silver_saved.columns

colunas_obrigatorias = [
    "id_rastreamento",
    "id_pedido_ecommerce",
    "codigo_rastreio",
    "id_transportadora",
    "status_entrega",
    "dt_evento",
    "observacao",
    "dias_em_transito",
    "bronze_source_file",
    "bronze_ingested_at",
    "silver_processed_at",
    "ano",
    "mes"
]

colunas_ausentes = [c for c in colunas_obrigatorias if c not in colunas_silver]

if colunas_ausentes:
    raise Exception(f"Erro: colunas obrigatórias ausentes na Silver: {colunas_ausentes}")

df_validacao_final = df_silver_saved.select(
    count("*").alias("total_linhas"),
    count(when(col("id_rastreamento").isNull(), True)).alias("id_rastreamento_nulo"),
    count(when(col("id_pedido_ecommerce").isNull(), True)).alias("id_pedido_ecommerce_nulo"),
    count(when(col("id_transportadora").isNull(), True)).alias("id_transportadora_nulo"),
    count(when(col("status_entrega").isNull(), True)).alias("status_entrega_nulo"),
    count(when(col("dt_evento").isNull(), True)).alias("dt_evento_nulo"),
    count(when(col("ano").isNull(), True)).alias("ano_nulo"),
    count(when(col("mes").isNull(), True)).alias("mes_nulo"),
    count(when(col("bronze_source_file").isNull(), True)).alias("bronze_source_file_nulo"),
    count(when(col("bronze_ingested_at").isNull(), True)).alias("bronze_ingested_at_nulo"),
    count(when(col("silver_processed_at").isNull(), True)).alias("silver_processed_at_nulo"),
    count(when(col("dias_em_transito").isNull(), True)).alias("dias_em_transito_nulo")
)

display(df_validacao_final)

validacao_final = df_validacao_final.collect()[0]

if validacao_final["id_rastreamento_nulo"] > 0:
    raise Exception("Erro: existem registros com id_rastreamento nulo na Silver.")

if validacao_final["dt_evento_nulo"] > 0:
    raise Exception("Erro: existem registros com dt_evento nulo na Silver.")

if validacao_final["ano_nulo"] > 0:
    raise Exception("Erro: existem registros com ano nulo na Silver.")

if validacao_final["mes_nulo"] > 0:
    raise Exception("Erro: existem registros com mes nulo na Silver.")

if validacao_final["bronze_source_file_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_source_file.")

if validacao_final["bronze_ingested_at_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_ingested_at.")

if validacao_final["silver_processed_at_nulo"] > 0:
    raise Exception("Erro: existem registros sem silver_processed_at.")

print("Validação OK: schema e qualidade final da Silver aprovados.")